# 03. Model Benchmarks, Threshold Optimization & SHAP Explainability
### Training 6 Algorithms, Evaluating Curves, and Calibrating Probabilities

We compare: Logistic Regression, Decision Tree, Random Forest, KNN, Gradient Boosting, and XGBoost.

In [ ]:
import sys, json
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt

# Load model comparison table from artifacts
comp_df = pd.read_csv('../artifacts/metrics/model_comparison.csv')
comp_df.sort_values(by='ROC-AUC', ascending=False)

## 1. Why Decision Threshold 0.230 Was Selected
Let's see what happens to Recall and Precision across different thresholds.

In [ ]:
from src.models.predict import NoShowPredictor
from src.models.evaluate import compute_classification_metrics

predictor = NoShowPredictor()
test_df = pd.read_csv('../data/processed/test_set.csv')
y_true = test_df['No-show'].to_numpy()
results = predictor.predict_batch(test_df.drop(columns=['No-show']))
y_prob = results['no_show_probability'].to_numpy()

m_default = compute_classification_metrics(y_true, y_prob, threshold=0.50, model_name='Default 0.50')
m_optimal = compute_classification_metrics(y_true, y_prob, threshold=0.23, model_name='Optimal 0.23')

print(f"Default 0.50 -> Recall: {m_default.recall:.1%} | Precision: {m_default.precision:.1%} | F1: {m_default.f1_score:.3f}")
print(f"Optimal 0.23 -> Recall: {m_optimal.recall:.1%} | Precision: {m_optimal.precision:.1%} | F1: {m_optimal.f1_score:.3f}")

## 2. Testing Live Inference with SHAP Attributions

In [ ]:
sample_patient = {
    'ScheduledDay': '2016-05-01T09:00:00Z',
    'AppointmentDay': '2016-05-18T00:00:00Z',
    'Age': 26,
    'Gender': 'F',
    'Neighbourhood': 'CENTRO',
    'Scholarship': 1,
    'Hipertension': 0,
    'Diabetes': 0,
    'Alcoholism': 0,
    'Handcap': 0,
    'prior_appointments': 1,
    'prior_no_shows': 1
}
res = predictor.predict_single(sample_patient, explain=True)
print(f"Prediction: {res.prediction_label} ({res.risk_category} Risk)")
print(f"Calibrated No-Show Probability: {res.no_show_probability:.1%}")

print("\nTop Factors Increasing Risk:")
for f in res.top_contributing_factors['top_risk_increasing_factors']:
    print(f"  - {f['feature']}: +{f['shap_value']:.3f}")